# RepairScopeX E06 — Reliability-Guided Intervention Routing

**Stand-alone CPU research notebook · version 1.0 · no repeated GPU inference.**

### What is genuinely implemented

1. Audit 648 frozen benchmark problems and every previously generated problem ID available in this package; prevent contamination of the proposed prospective test set.
2. Load the **actual E08 and E09 calibration results** from Google Drive or ZIP uploads; verify trial identity and full action coverage. Missing checkpoints are reported as **pending**, never converted to invented zeros.
3. Fit a low-capacity, cost-aware intervention selector on completed calibration trials; use **leave-one-problem-out** checks and compare with fixed policies. Analyses remain *exploratory*, due to very small, differently selected cohorts and nonidentical action menus.
4. Pre-register 36 unused held-out problems with a common three-action protocol (finalize, selective remask, regenerate). The output manifest omits gold answers and is **not** an already-executed test.
5. Optionally evaluate future **genuinely prospective** results only after all three actions are available for each held-out source generation and the policy was locked first.

**Scientific limits:** E08 studies previously *gold-screened definite failures* and E09 studies previously *gold-screened ambiguous outputs*. Their action menus differ. These results cannot establish a deployment-ready three-way router or Q1 superiority. The E06 notebook explicitly refuses to report confirmatory gain without a complete untouched, common-action comparison. It does **not** retrain a model or rerun any previously completed trials.

**Execution:** CPU Colab → Runtime → Run all. Rerun later after E08/E09 have results. New results are read, not recomputed.

In [ ]:
import sys, csv, io, os, json, zlib, base64, zipfile, hashlib, math, statistics, random, re, datetime
from pathlib import Path
from collections import defaultdict, Counter

# Optional explicit paths to E08/E09 finished ZIP checkpoints; otherwise auto-discover Drive folders.
E08_ZIP = ""     # e.g. "/content/RSX_E08_Checkpoint.zip"
E09_ZIP = ""     # e.g. "/content/RSX_E09_Checkpoint.zip"
PROSPECTIVE_ZIP = "" # optional, ONLY from a locked future E06 held-out experiment
GOOGLE_DRIVE_ENABLED = True

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
if IS_COLAB and GOOGLE_DRIVE_ENABLED:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print('Google Drive unavailable; using local folder:', repr(exc))

DRIVE_ROOT = Path('/content/drive/MyDrive/RepairScopeX')
OUT = (DRIVE_ROOT/'E06_Router_V1') if DRIVE_ROOT.is_dir() else (Path('/content/E06_Router_V1') if Path('/content').exists() else Path('/mnt/data/E06_Router_V1'))
OUT.mkdir(parents=True, exist_ok=True)
print('E06 output folder:', OUT)

def sha(o):
    return hashlib.sha256(json.dumps(o, sort_keys=True, ensure_ascii=False, separators=(',',':')).encode()).hexdigest()

def atomic_json(p, obj):
    p=Path(p);tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True, allow_nan=False),encoding='utf-8')
    os.replace(tmp,p)

def frozen_json(p, obj):
    p=Path(p)
    if p.exists():
        if json.loads(p.read_text(encoding='utf-8')) != obj:
            raise RuntimeError(f'Frozen research artifact changed: {p}. Preserve the original; use a new protocol version.')
    else: atomic_json(p,obj)

def write_csv(path, rows, cols):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w', newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=cols,extrasaction='ignore');w.writeheader();w.writerows(rows)
    os.replace(tmp,path)

def wilson(s,n,z=1.95996398454):
    if n == 0:return (None,None)
    p=s/n;d=1+z*z/n;c=(p+z*z/(2*n))/d
    a=z/d*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return (round(max(0,c-a),5),round(min(1,c+a),5))

In [ ]:
# Immutable local evidence and original frozen problem bank are embedded.
# User does NOT need to upload prior V1, NB29H, P1 or the 648-problem bank.
DATA=json.loads(zlib.decompress(base64.b64decode("...")))
BANK=DATA['bank']['problems']
assert len(BANK)==648 and len({p['problem_id'] for p in BANK})==648
assert len(DATA['E08_cases'])==6 and len(DATA['E09_cases'])==12
BANK_BY_ID={p['problem_id']:p for p in BANK}
USED=set(pid for ids in DATA['prior_touched'].values() for pid in ids)
assert set(c['problem']['problem_id'] for c in DATA['E08_cases']+DATA['E09_cases']).issubset(USED)
assert not ({c['problem']['problem_id'] for c in DATA['E08_cases']} & {c['problem']['problem_id'] for c in DATA['E09_cases']})
assert all(c['problem']['split']=='calibration' for c in DATA['E08_cases']+DATA['E09_cases'])
assert len(DATA['P1_records'])==48
print('Verified frozen problems', len(BANK),'previously exposed problem IDs',len(USED))
print('Original E08/E09 source cases:',len(DATA['E08_cases']),len(DATA['E09_cases']))

# These source observables are available without the correct answer, score or gold failure label.
ANSWER_FIELD=re.compile(r'(?im)^\s*(?:FINAL_ANSWER|ANSWER)\s*:\s*[-+]?(?:\d+(?:\.\d*)?|\.\d+)')
BOX_FIELD=re.compile(r'\\boxed\s*\{\s*[-+]?\d+(?:\.\d*)?\s*\}')

def observable_features(source):
    text=str(source.get('output') or source.get('repaired_output') or '')
    # No reading source['evaluation'], problem['target'], or source gold correctness.
    return {
        'has_explicit_final':bool(ANSWER_FIELD.search(text) or BOX_FIELD.search(text)),
        'ended_at_length':str(source.get('stop_reason','')).lower() in ('length','max_length','max_new_tokens'),
        'has_arithmetic':bool(re.search(r'\d\s*[-+*/×÷]\s*\d',text)),
        'output_characters':len(text),
    }

# Same features despite changing target or grader: a regression test against gold-label leakage.
s=DATA['E08_cases'][0]['source']
modified=dict(s);modified['evaluation']={'success':not s['evaluation'].get('success')};modified['target']=123456789
assert observable_features(modified)==observable_features(s)
print('Target-label leakage regression: PASS')

## A. Prospective manifest — independently unused held-out examples

The manifest selects 12 unused examples per task family from the original bank using a frozen deterministic hash. It excludes every problem exposed in the historical V1, NB29G, NB29H, E02, E03, P1, E08, and E09 evidence. **No model runs are performed here.** New source generations and all three intervention arms will still need to be collected prospectively under a separately pinned and equal-budget GPU protocol.

In [ ]:
SPEC={
  'study':'RepairScopeX_E06_router_v1',
  'calibration_only':['E08: repair-locality','E09: finalization'],
  'router_observables':['has_explicit_final','ended_at_length','has_arithmetic','output_characters'],
  'selection_seed':'RSX-E06-prospective-precommitted-v1',
  'prospective_split':'heldout',
  'prospective_per_family':12,
  'prospective_actions':['finalize','selective_remask','regenerate'],
  'prospective_source_generation':'one fixed initial generation per problem before interventions',
  'prospective_action_budget_tokens':96,
  'prospective_required_fields':['trial_id','problem_id','split','action','source_trial_id','source_output','source_stop_reason','success','loop_token_passes','protocol_hash'],
  'allow_adaptive_policy_refit_on_prospective':False,
  'method_note':'Action menus differ between E08 and E09; calibrations are exploratory, confirmatory claims require identical tri-arm prospective evaluations.',
}
SPEC_HASH=sha(SPEC)
frozen_json(OUT/'E06_STUDY_SPEC.json',{'spec':SPEC,'spec_hash':SPEC_HASH})
future=[]
for kind in ('arithmetic','countdown','multistep'):
    pool=[p for p in BANK if p['split']=='heldout' and p['kind']==kind and p['problem_id'] not in USED]
    pool.sort(key=lambda p: hashlib.sha256((SPEC['selection_seed']+'|'+p['problem_id']).encode()).hexdigest())
    assert len(pool)>=12, f'Not enough independent held-out problems for {kind}'
    for p in pool[:12]:
        future.append({'problem_id':p['problem_id'],'kind':kind,'split':'heldout',
            'prompt_sha256':hashlib.sha256(p['prompt'].encode()).hexdigest(),
            'problem_fingerprint':p['problem_fingerprint']})
assert len(future)==36 and not {x['problem_id'] for x in future}&USED
manifest={'spec_hash':SPEC_HASH,'problems':future}
frozen_json(OUT/'E06_BLINDED_HOLDOUT_MANIFEST.json',manifest)
write_csv(OUT/'E06_BLINDED_HOLDOUT_MANIFEST.csv',future,['problem_id','kind','split','prompt_sha256','problem_fingerprint'])
print('Prospective manifest frozen: 36 NEW held-out problems; 12 per task family; 108 planned action evaluations.')
print('WARNING: these are future plans, NOT observed results.')

## B. Safe import of E08 and E09 calibration results

The notebook reads the existing **checkpoints**, never invokes a diffusion model and never repeats a trial. Empty or partial directories stay pending. It verifies the exact precommitted trial IDs, original parent protocol hashes, expected calibration problem sets, decoding arms and costs. A source ZIP can be supplied in the configuration above; otherwise Google Drive is checked automatically. E08 requires 72/72 and E09 requires 96/96 for a final exploratory calibration.

In [ ]:
SOURCE_BY_STUDY={
    'E08':DATA['E08_cases'],
    'E09':DATA['E09_cases'],
}
E08_ACTIONS=('global','tail25','tail50','tail75','answer_locus','first_math')
E08_DECODES=('left1','entropy')
E09_ACTIONS=('direct_one_line','contextual_finalizer','prefix_continuation','fresh_resolve')
E09_BUDGETS=(96,192)

def read_json_records(folder=None, zip_path=None):
    if zip_path:
        p=Path(zip_path)
        if not p.is_file():raise FileNotFoundError(f'ZIP not found: {p}')
        with zipfile.ZipFile(p) as z:
            names=[n for n in z.namelist() if n.startswith('records/') and n.endswith('.json')]
            return [json.loads(z.read(n)) for n in names],f'zip:{p.name}'
    folder=Path(folder)
    if not folder.is_dir():return [],'not available'
    return [json.loads(p.read_text(encoding='utf-8')) for p in sorted((folder/'records').glob('*.json'))],str(folder)

CHECKPOINTS={
 'E08':(E08_ZIP,DRIVE_ROOT/'E08_RepairLocality_V1'),
 'E09':(E09_ZIP,DRIVE_ROOT/'E09_Finalization_V1'),
}
INPUTS={}
counts={}
for study,(zip_path,folder) in CHECKPOINTS.items():
    records,source=read_json_records(folder=folder,zip_path=zip_path or None)
    ids=[r.get('trial_id') for r in records]
    if len(ids)!=len(set(ids)) or any(not t for t in ids):raise RuntimeError(f'{study}: duplicate or missing trial ID')
    valid_pids={c['problem']['problem_id'] for c in SOURCE_BY_STUDY[study]}
    expected=set()
    for pid in valid_pids:
        if study=='E08':expected.update(f'E08|{pid}|{a}|{d}|62083' for a in E08_ACTIONS for d in E08_DECODES)
        else:expected.update(f'E09|{pid}|{a}|{b}|62091' for a in E09_ACTIONS for b in E09_BUDGETS)
    for r in records:
        assert r['trial_id'] in expected, f'{study}: unexpected trial {r["trial_id"]}'
        assert r['problem_id'] in valid_pids and r['split']=='calibration'
        assert r['evaluation'].get('success') in (True,False)
        assert isinstance(r.get('loop_token_passes'),int) and r['loop_token_passes']>=0
        if study=='E08':
            assert r['action'] in E08_ACTIONS and r['decode'] in E08_DECODES
            assert r.get('exact_prefix_invariant') is True
        else:
            assert r['action'] in E09_ACTIONS and r['budget'] in E09_BUDGETS
            assert r.get('preservation_invariant') is True
    # Every saved record belongs to a single frozen protocol.
    hashes={r['protocol_hash'] for r in records}
    assert len(hashes)<=1, f'{study}: mixed protocol hashes'
    # If available, validate the on-disk protocol file rather than trusting record labels alone.
    if not zip_path and (folder/'protocol.json').exists():
        protocol=json.loads((folder/'protocol.json').read_text())
        if hashes:assert protocol['hash'] in hashes
        condition=protocol.get('condition',{})
        assert condition.get('parent_h_protocol')==DATA['E08_parent_hash' if study=='E08' else 'E09_parent_hash']
    if zip_path:
        with zipfile.ZipFile(zip_path) as z:
            matches=[n for n in z.namelist() if n.endswith('protocol.json')]
            if matches:
                protocol=json.loads(z.read(matches[0]));condition=protocol.get('condition',{})
                assert condition.get('parent_h_protocol')==DATA['E08_parent_hash' if study=='E08' else 'E09_parent_hash']
                if hashes:assert protocol['hash'] in hashes
    INPUTS[study]=records
    counts[study]={'complete':len(records),'expected':len(expected),'source':source}
    print(study, len(records),'/',len(expected),source)
atomic_json(OUT/'E06_CALIBRATION_INPUT_STATUS.json',counts)
CALIBRATION_COMPLETE=all(counts[k]['complete']==counts[k]['expected'] for k in ('E08','E09'))
print('Both calibration matrices complete:', CALIBRATION_COMPLETE)

## C. Cost-aware routing calibration and leave-one-problem-out audit

**Guardrail:** Fitting is done only if both E08 and E09 experimental matrices are complete. E08 and E09 use disjoint calibration questions **but not the same action menus**. Therefore, this is an *exploratory family-specific selector*, not a validated three-way classifier. All comparisons use unique problems as sampling units; repeated decoding arms are not claimed as new independent problems. If a checkpoint is missing, only planning artifacts are exported.

In [ ]:
def action_name(study,rec):
    return (rec['action']+'|'+rec['decode']) if study=='E08' else (rec['action']+'|'+str(rec['budget']))

def build_matrix(study,records):
    cases={c['problem']['problem_id']:c for c in SOURCE_BY_STUDY[study]}
    matrix={p:{} for p in cases}
    for r in records:
        aid=action_name(study,r);pid=r['problem_id']
        if aid in matrix[pid]:raise AssertionError('Duplicate action per problem: '+pid+' '+aid)
        matrix[pid][aid]={
            'correct':int(r['evaluation']['success']),
            'forward_passes':int(r['loop_token_passes']),
            'wall_seconds':float(r.get('wall_seconds',0)),
            'record_id':r['trial_id'],
        }
    return matrix

def select_policy(training,lam=0.15):
    # Within training folds only, define scale using global/fresh action costs.
    items=[v for arms in training.values() for act,v in arms.items() if act.startswith(('global|','fresh_resolve|'))]
    if not items:items=[v for arms in training.values() for v in arms.values()]
    cost_scale=max(1.0,statistics.median(v['forward_passes'] for v in items))
    actions=sorted(set.intersection(*(set(arms) for arms in training.values())))
    if not actions: raise AssertionError('No common calibration interventions')
    rows=[]
    for a in actions:
        hits=[arms[a]['correct'] for arms in training.values()]
        prices=[arms[a]['forward_passes'] for arms in training.values()]
        score=statistics.mean(hits)-lam*statistics.mean(prices)/cost_scale
        rows.append({'action':a,'utility':score,'correct_rate':statistics.mean(hits),'mean_passes':statistics.mean(prices)})
    winner=sorted(rows,key=lambda r:(-r['utility'],-r['correct_rate'],r['mean_passes'],r['action']))[0]
    return winner,rows

if CALIBRATION_COMPLETE:
    DETAIL=[];FROZEN={};LOPO=[]
    for study in ('E08','E09'):
        mat=build_matrix(study,INPUTS[study]); expected_arms=(12 if study=='E08' else 8)
        assert all(len(v)==expected_arms for v in mat.values())
        assert len(mat)==(6 if study=='E08' else 12)
        winner,details=select_policy(mat,lam=0.15)
        FROZEN[study]={'action':winner['action'],'training_unique_problems':len(mat),'candidate_stats':details}
        DETAIL.extend([dict(study=study,**r) for r in details])
        for pid in sorted(mat):
            train={k:v for k,v in mat.items() if k!=pid}
            chosen,_=select_policy(train,lam=0.15)
            result=mat[pid][chosen['action']]
            LOPO.append({'study':study,'problem_id':pid,'selected_action':chosen['action'],
                'correct':result['correct'],'forward_passes':result['forward_passes']})
    write_csv(OUT/'E06_CALIBRATION_ACTION_TABLE.csv',DETAIL,['study','action','utility','correct_rate','mean_passes'])
    write_csv(OUT/'E06_LEAVE_ONE_PROBLEM_OUT.csv',LOPO,['study','problem_id','selected_action','correct','forward_passes'])
    # Observable router: the textual final-answer marker alone chooses which family policy to consult.
    # Conversion to action families is a proposal, NOT a measured cross-family comparison.
    e08=FROZEN['E08']['action'].split('|')[0];e09=FROZEN['E09']['action'].split('|')[0]
    repair_family='regenerate' if e08=='global' else 'selective_remask'
    ambiguity_family='regenerate' if e09=='fresh_resolve' else 'finalize'
    router={'if_final_answer_present':repair_family,
            'otherwise':ambiguity_family,
            'features_used':['has_explicit_final'],
            'never_uses_gold_outcome':True,
            'calibration_prevalence_note':'gold-screened calibration subsets; class prevalences not transferable'}
    POLICY={'study_spec_hash':SPEC_HASH,'calibration_source_protocol_hashes':{
           s:sorted({r['protocol_hash'] for r in INPUTS[s]})[0] for s in ('E08','E09')},
           'family_specific_winners':FROZEN,'router':router,
           'selection_objective':'success - 0.15 * forward_passes / training_median_reference_cost',
           'interpretation':'prospective candidate policy ONLY; no cross-family effectiveness established'}
    frozen_json(OUT/'E06_FROZEN_CANDIDATE_POLICY.json',POLICY)
    report={s:{'n_problems':len([r for r in LOPO if r['study']==s]),
               'lopo_successes':sum(r['correct'] for r in LOPO if r['study']==s),
               'wilson_95':wilson(sum(r['correct'] for r in LOPO if r['study']==s),len([r for r in LOPO if r['study']==s])),
               'selected_action':FROZEN[s]['action']} for s in ('E08','E09')}
    atomic_json(OUT/'E06_CALIBRATION_REPORT.json',report)
    print('EXPLORATORY CALIBRATION COMPLETE: policy frozen; LOPO tested; NOT held-out validation.')
    print(json.dumps(report,indent=2))
else:
    print('CALIBRATION PENDING — neither missing-trial counts nor invented outcomes are used.')
    print('Run E08/E09 from their OWN checkpoint when convenient; rerun E06 on CPU later.')
    print('Current status:',counts)

## D. Independent prospective evaluation gate (optional)

A Q1-level effectiveness claim needs a new full common-action evaluation: same 36 unexposed held-out questions, one **frozen source generation per problem** and all three actions evaluated at comparable budgets. Do not choose new held-out cases based on observed failures. The source generation should not contain a gold label in the router input. Collect all arms with frozen configuration, then pass the outcome ZIP into this notebook.

This section refuses incomplete experiments, duplicated source/action pairs, missing costs, mixed source generations or mismatched protocol hashes. A **locked** E06 policy must already exist before evaluation. Prospective rows do not train or update the policy.

In [ ]:
TEMPLATE=[]
for p in future:
    for action in SPEC['prospective_actions']:
        TEMPLATE.append({'trial_id':'','problem_id':p['problem_id'],'split':'heldout','action':action,
            'source_trial_id':'','source_output':'','source_stop_reason':'',
            'success':'','loop_token_passes':'','protocol_hash':SPEC_HASH})
write_csv(OUT/'E06_PROSPECTIVE_RESULT_SCHEMA_TEMPLATE.csv',TEMPLATE,SPEC['prospective_required_fields'])

if PROSPECTIVE_ZIP:
    if not (OUT/'E06_FROZEN_CANDIDATE_POLICY.json').exists():
        raise RuntimeError('A candidate policy must be locked before prospective evaluation.')
    with zipfile.ZipFile(PROSPECTIVE_ZIP) as z:
        names=[n for n in z.namelist() if n.endswith('.json') and ('records/' in n)]
        new=[json.loads(z.read(n)) for n in names]
    expected_pairs={(p['problem_id'],a) for p in future for a in SPEC['prospective_actions']}
    got_pairs=[(r['problem_id'],r['action']) for r in new]
    if len(got_pairs)!=len(set(got_pairs)) or set(got_pairs)!=expected_pairs:
        raise RuntimeError(f'Prospective study INCOMPLETE: {len(set(got_pairs))}/{len(expected_pairs)} pairs. No confirmatory metric generated.')
    pids={p['problem_id'] for p in future}
    base_sources={}
    for r in new:
        assert r['problem_id'] in pids and r['split']=='heldout'
        assert r['protocol_hash']==SPEC_HASH
        assert r['success'] in (True,False)
        assert isinstance(r['loop_token_passes'],int) and r['loop_token_passes']>=0
        src=(r['source_trial_id'],r['source_output'],r['source_stop_reason'])
        pid=r['problem_id']
        if pid in base_sources:assert base_sources[pid]==src, 'Intervention arms must share the same natural source generation'
        base_sources[pid]=src
    policy=json.loads((OUT/'E06_FROZEN_CANDIDATE_POLICY.json').read_text())
    pmap={(r['problem_id'],r['action']):r for r in new}
    scored=[]
    for p in future:
        pid=p['problem_id'];source=base_sources[pid]
        features=observable_features({'output':source[1],'stop_reason':source[2]})
        chosen=(policy['router']['if_final_answer_present'] if features['has_explicit_final'] else policy['router']['otherwise'])
        actual=pmap[(pid,chosen)]
        fixed=pmap[(pid,'regenerate')]
        scored.append({'problem_id':pid,'chosen_action':chosen,'router_success':int(actual['success']),
           'regenerate_success':int(fixed['success']),'router_forward_passes':actual['loop_token_passes'],
           'regenerate_forward_passes':fixed['loop_token_passes']})
    write_csv(OUT/'E06_PROSPECTIVE_COMPARISON.csv',scored,['problem_id','chosen_action','router_success','regenerate_success','router_forward_passes','regenerate_forward_passes'])
    difference=[r['router_success']-r['regenerate_success'] for r in scored]
    rng=random.Random(62006)
    boot=[statistics.mean(rng.choices(difference,k=len(difference))) for i in range(4000)]
    boot.sort();lo=boot[int(.025*len(boot))];hi=boot[int(.975*len(boot))]
    conclusion={'n_unique_heldout_problems':len(scored),'router_correct':sum(r['router_success'] for r in scored),
       'regenerate_correct':sum(r['regenerate_success'] for r in scored),
       'paired_success_difference':statistics.mean(difference),'problem_bootstrap_ci95':[lo,hi],
       'mean_router_forward_passes':statistics.mean(r['router_forward_passes'] for r in scored),
       'mean_regenerate_forward_passes':statistics.mean(r['regenerate_forward_passes'] for r in scored),
       'interpretation':'Paired held-out comparison conditional on prospectively submitted complete matching records; not a guarantee of significance'}
    atomic_json(OUT/'E06_PROSPECTIVE_EVALUATION.json',conclusion)
    print('PROSPECTIVE GATE PASSED. Independent outcomes summarized:',json.dumps(conclusion,indent=2))
else:
    print('No new independently collected common-action held-out results provided. Confirmatory evaluation: NOT RUN.')
    print('Schema template generated; no success rate fabricated.')

## E. Export and final audit

The ZIP includes the latest status, study specification, never-before-run blinded manifest, and (only if the true results are present) exploratory fitted policy and confirmatory held-out analysis. It does **not** contain regenerated model outputs.

In [ ]:
def build_export():
    archive=OUT.parent/'RepairScopeX_E06_Artifacts.zip'
    # Do not add the archive to itself or include temporary files.
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for file in sorted(OUT.rglob('*')):
            if file.is_file() and not file.name.endswith('.tmp'):
                z.write(file,file.relative_to(OUT))
    return archive

status={'E06_design_complete':True,'E08_calibration_completed':counts['E08']['complete'],
    'E08_calibration_target':72,'E09_calibration_completed':counts['E09']['complete'],
    'E09_calibration_target':96,'policy_locked':(OUT/'E06_FROZEN_CANDIDATE_POLICY.json').exists(),
    'prospective_n_planned_problems':36,
    'prospective_confirmatory_evaluation_done':(OUT/'E06_PROSPECTIVE_EVALUATION.json').exists(),
    'no_prior_generations_rerun':True,
    'conclusion':'Method skeleton and prospective plan prepared; final performance not established unless independently supplied held-out results pass all checks.'}
atomic_json(OUT/'E06_EXECUTION_STATUS.json',status)
print(json.dumps(status,indent=2))
archive=build_export();print('OUTPUT ZIP:',archive,'size (KiB):',round(archive.stat().st_size/1024,1))
if IS_COLAB:
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as exc:print('Automatic download unavailable:',repr(exc))